In [13]:
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from pathlib import Path
import geopandas as gp
import spatialdata as sd
from tqdm import tqdm
from shapely.geometry import box
import geopandas as gpd
from spatialdata.models import ShapesModel, Labels2DModel, Image2DModel
from src.utils import read_obs
from datetime import datetime

In [14]:
current_datetime = datetime.now()
formatted_date = current_datetime.strftime("%Y-%m-%d")

In [2]:
sdata_paths = list(Path("/data/sdata_ptau_1").glob("*.zarr")) + list(Path("/data/sdata_ptau_2").glob("*.zarr")) 
dfs = {"cell_dfs":{}, "tau_dfs":{}, "plaque_dfs":{}}
for path in sdata_paths:
    barcode = path.stem.split("_")[0]
    sdata = sd.read_zarr(path)
    dfs['cell_dfs'][barcode] = sd.transform(sdata['cell_boundaries'], to_coordinate_system='global')
    dfs['tau_dfs'][barcode] = sd.transform(sdata['cellular_tau_boundaries'], to_coordinate_system='global')
    dfs['plaque_dfs'][barcode] = sd.transform(sdata['plaque_boundaries'], to_coordinate_system='global')

/tmp/ipykernel_26566/2266186248.py:5: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
no parent found for <ome_zarr.reader.Label object at 0x7f944cdf0920>: None
no parent found for <ome_zarr.reader.Label object at 0x7f944cddfe60>: None
no parent found for <ome_zarr.reader.Label object at 0x7f944cd5a540>: None
no parent found for <ome_zarr.reader.Label object at 0x7f944c233890>: None
no parent found for <ome_zarr.reader.Label object at 0x7f944c2442c0>: None
/tmp/ipykernel_26566/2266186248.py:5: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(path)
no parent found for <ome_zarr.reader.Label object at 0x7f944cf45790>: None
no parent found for <ome_zarr.reader.Label object at 0x7f944cd9f7d0>: None
no parent found for <om

In [4]:
tau_df = pd.concat(dfs['tau_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})
plaque_df = pd.concat(dfs['plaque_dfs']).drop('label', axis = 1).reset_index().rename(columns = {"level_0":"barcode"})

In [6]:
obs = read_obs("/root/capsule/data/combined_adata/CaH_Xenium.2026-01-07.h5ad")

In [7]:
cell_df = pd.concat(dfs['cell_dfs']).reset_index().rename(columns = {'level_0':'barcode', 'level_1':'label'})

In [8]:
cell_df.index = cell_df['label'] + "_" + cell_df['barcode']

In [10]:
obs.index = obs['cell_id'].astype(str) + "_" + obs['barcode'].astype(str)

In [11]:
cluster_columns = ['Neighborhood', 'Subclass', 'Supertype']

In [12]:
cell_df = cell_df.merge(obs[cluster_columns], left_index = True, right_index = True, how = "left")

In [15]:
tau_df.to_csv(f"/results/seaad_cah_tau_polygons.{formatted_date}.csv")

In [16]:
plaque_df.to_csv(f"/results/seaad_cah_plaque_polygons.{formatted_date}.csv")

In [17]:
cell_df.to_csv(f"/results/seaad_cah_cell_segmentation_polgyons.{formatted_date}.csv")